In [1]:
import pandas as pd
import pyarrow.parquet as pq
import numpy as np
import time

# Update this path to wherever your 8GB parquet file actually lives on your local machine
GRID_FILE_PATH = "../data/HADES_grid/hades_processed_grid.parquet"

# Constants
M_J = 1.898e27
R_J = 69911000

In [2]:
# This reads ONLY the metadata, not the 8GB of data
parquet_file = pq.ParquetFile(GRID_FILE_PATH)

print(f"Total rows in file: {parquet_file.metadata.num_rows:,}")
print(f"Total row groups (chunks): {parquet_file.metadata.num_row_groups}")
print(f"Number of columns: {parquet_file.metadata.num_columns}")

# Let's peek at just the column names
print("\nFirst 15 columns:")
print(parquet_file.schema.names[:15])

Total rows in file: 94,189
Total row groups (chunks): 1
Number of columns: 46

First 15 columns:
['mass', 'Req', 'T_int', 'T_irr', 'Met', 'core', 'f_sed', 'kzz', 'S_physical', 'dsdt', 'Lint', 'element', 'element', 'element', 'element']


In [3]:
start_time = time.time()

# The columns we care about for the ML model
needed_columns = [
    'mass', 'T_int', 'T_irr', 'Met', 'core', 'f_sed', 'kzz', 
    'Req', 'S_physical', 'dsdt'
]

# We must use the raw units that exist IN the file for the filters
mass_threshold_kg = 5 * M_J

# Predicate pushdown: filter the data BEFORE it enters RAM
filters = [
    ('T_int', '<', 500),
    ('mass', '<=', mass_threshold_kg)
]

print("Loading filtered slice...")
df_slice = pd.read_parquet(
    GRID_FILE_PATH, 
    engine='pyarrow',
    columns=needed_columns,
    filters=filters
)

print(f"Loaded {len(df_slice):,} rows in {time.time() - start_time:.2f} seconds!")
display(df_slice.head())

Loading filtered slice...
Loaded 10,969 rows in 0.07 seconds!


,mass,T_int,T_irr,Met,core,f_sed,kzz,Req,S_physical,dsdt
0,1.047526e+25,218.989512,92.51,-1.10,1.736522,2.83,10.87,3.139584e+07,62672.316384,-1.634536e-09
1,1.341753e+25,213.659580,50.11,-1.92,2.224273,3.02,3.04,2.405008e+07,58781.649198,-8.990034e-10
2,1.254506e+25,255.866370,103.98,-0.82,2.079640,3.04,7.00,3.443989e+07,64451.616160,-3.772766e-09
3,2.213393e+25,419.004949,217.62,-1.16,3.669222,3.38,1.32,5.192560e+07,70101.477020,-2.158689e-08
4,2.377620e+25,169.341157,120.91,-1.22,3.941466,4.32,3.38,2.143983e+07,56061.802383,-3.060911e-10


In [4]:
# Convert to Jupiter units
df_slice['mass_Mj'] = df_slice['mass'] / M_J
df_slice['Req_Rj'] = df_slice['Req'] / R_J

# Target variable for the ODE
df_slice['abs_log_dsdt'] = np.log10(np.abs(df_slice['dsdt']))

# Drop any NaNs
df_slice = df_slice.dropna().reset_index(drop=True)

print("Data preprocessed successfully. Final shape:", df_slice.shape)
display(df_slice[['mass_Mj', 'T_int', 'abs_log_dsdt']].head())

Data preprocessed successfully. Final shape: (10945, 13)


,mass_Mj,T_int,abs_log_dsdt
0,0.005519,218.989512,-8.786605
1,0.007069,213.659580,-9.046239
2,0.006610,255.866370,-8.423340
3,0.011662,419.004949,-7.665810
4,0.012527,169.341157,-9.514149
